<div >
<img src = "figs/ans_banner_1920x200.png" />
</div>

# Caso-taller:  Analizando el Delito en Chicago

En este caso-taller vamos a utilizar datos geográficos y estimación de densidad de kernel para analizar delitos en Chicago. Esta ciudad es muy famosa no sólo por haber sido el hogar del mafioso Al Capone, sino también por sus altas tasas de delitos. 

Para este taller obtuve datos del portal de la [ciudad de Chicago](https://www.chicago.gov/city/en/dataset/crime.html). La base de datos fue traducida y modificada para nuestras necesidades. Esta contiene todos los homicidios y robos que sucedieron entre el 1 de junio y el 31 de agosto de 2019.


## Instrucciones generales

1. Para desarrollar el *cuaderno* primero debe descargarlo.

2. Para responder cada inciso deberá utilizar el espacio debidamente especificado.

3. La actividad será calificada sólo si sube el *cuaderno* de jupyter notebook con extensión `.ipynb` en la actividad designada como "entrega calificada por el personal".

4. El archivo entregado debe poder ser ejecutado localmente por el tutor. Sea cuidadoso con la especificación de la ubicación de los archivos de soporte, guarde la carpeta de datos en el mismo `path` de su cuaderno, por ejemplo: `data`.

## Integrantes

Grupo 29

|**Nombre**|**Email**|
|---|---|
|Jaramillo Gómez Simon|s.jaramillo3@uniandes.edu.co|
|Mendoza Canales Hubert Ronald|h.mendozac@uniandes.edu.co|
|Montero Ramirez Daniel Eduardo|de.montero@uniandes.edu.co|

## Desarrollo


#### Config

In [2]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

# python
import shutil
import zipfile
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import session_info

# tools
from pathlib import Path
from inspect import cleandoc
from dataclasses import dataclass

# stats
import statsmodels.api as sm
from scipy import stats

# Geo
import folium
import geojsoncontour
import geopandas as gpd
from pyrosm import OSM, get_data
from geopy.distance import geodesic
from shapely.ops import nearest_points


# sklearn
from sklearn import model_selection
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# view
from IPython.display import IFrame
from bokeh.models import ColumnDataSource
from bokeh.plotting import figure, output_notebook, show
from bokeh.tile_providers import CARTODBPOSITRON, get_provider


# typings
from typing import List, Tuple, Dict, Any, Union, Optional

# setup
plt.style.use('seaborn')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('max_colwidth', None)

# decimals
np.set_printoptions(precision=6)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


##### Información de Sesión

In [3]:
session_info.show(html=False)

-----
bokeh               2.4.3
folium              0.14.0
geojsoncontour      NA
geopandas           0.10.2
geopy               2.4.0
matplotlib          3.3.1
numpy               1.19.1
pandas              1.1.1
pyrosm              NA
scipy               1.7.3
seaborn             0.12.2
session_info        1.0.0
shapely             1.8.5
sklearn             0.23.2
statsmodels         0.13.5
-----
IPython             7.34.0
jupyter_client      7.4.9
jupyter_core        4.12.0
jupyterlab          2.1.1
notebook            6.0.3
-----
Python 3.7.17 (default, Jun 13 2023, 16:36:15) [GCC 8.3.0]
Linux-5.15.49-linuxkit-x86_64-with-debian-10.13
-----
Session information updated at 2025-09-20 15:23


### utils

In [8]:
# utils
# rutas absolutas: agnostico al sistema operativo
here: Path = Path.cwd().absolute()
data: Path = here / 'data'

chicago_areas_zip: Path = data / 'Areas_comunitarias_Chicago.zip'
chicago_areas_data: Path = data / 'Areas_comunitarias_Chicago'
chicago_crime_data: Path = data / 'Chicago_delitos_verano_2019.csv'

### 1.Carga de datos 

#### 1.1. Delitos

En la carpeta `data` se encuentra el archivo `Chicago_delitos_verano_2019.csv` cargue estos datos en su *cuaderno*. Describa brevemente el contenido de la base.

In [21]:
# Utilice este espacio para escribir el código.
@dataclass(frozen=True)
class Constant:
    SEP:str = ','
    ENCODING:str = 'utf-8'
    EMPTY_STR:str = ''
    SPACE_STR:str = ' '
    ONE:int = 1
    ZERO:int = 0
    
if not chicago_crime_data.exists():
    raise FileNotFoundError(cleandoc(f'''
        El archivo {chicago_crime_data} no existe.
        y colóquelo en el directorio {data}
    '''))
    
setup:Dict = dict(sep=Constant.SEP, encoding=Constant.ENCODING)

chicago_crime_df = pd.read_csv(chicago_crime_data, **setup)
chicago_crime_df.columns = chicago_crime_df.columns.str.strip()

print(chicago_crime_df.info())
display(chicago_crime_df.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17747 entries, 0 to 17746
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fecha                 17747 non-null  object 
 1   tipo_crimen           17747 non-null  object 
 2   nro_area_comunitaria  17747 non-null  int64  
 3   lat                   17747 non-null  float64
 4   lon                   17747 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 693.4+ KB
None


,fecha,tipo_crimen,nro_area_comunitaria,lat,lon
0,2019-06-01T05:07:00Z,homicidio,23,41.897950,-87.728625
1,2019-06-01T10:09:00Z,homicidio,71,41.753272,-87.648963
2,2019-06-01T12:46:00Z,homicidio,25,41.877622,-87.750728
3,2019-06-01T11:35:00Z,homicidio,16,41.960145,-87.699654
4,2019-06-02T09:39:00Z,homicidio,37,41.804773,-87.633256


(Utilice este espacio para describir su procedimiento)

#### 1.2. Barrios de Chicago

También en la carpeta `data` se encuentran los archivos con los polígonos de las áreas comunitarias en un archivo comprimido llamado `Areas_comunitarias_Chicago.zip`. Genere un mapa interactivo con un popup con el nombre del area comunitaria.

In [ ]:
# Utilice este espacio para escribir el código.
# helper function
def preview_geoms(
        df: gpd.GeoDataFrame, 
        n: int = 5, 
        width: int = 80,
        colname: str = "geometry"
    ) -> pd.DataFrame:
    """
    Muestra las primeras n filas de un GeoDataFrame truncando 
    la columna 'geometry' por defecto.
    
    Parameters
    ----------
    df : gpd.GeoDataFrame
        GeoDataFrame a visualizar.
    n : int, optional
        Número de filas a mostrar (default=5).
    width : int, optional
        Máxima longitud de la representación de geometría (default=80).
    
    Returns
    -------
    pd.DataFrame
        DataFrame con geometría truncada para previsualización.
    """
    df_copy = df.copy()
    if colname in df_copy.columns:
        df_copy[colname] = df_copy[colname].astype(str).str.slice(0, width) + "..."
    return df_copy.head(n)

# validar si esta descomprimido y si existe el archivo zip
if not chicago_areas_data.exists() or not any(chicago_areas_data.iterdir()):
    with zipfile.ZipFile(chicago_areas_zip, "r") as zip_ref:
        zip_ref.extractall(chicago_areas_data)
        
chicago_area_df: pd.DataFrame = gpd.read_file(chicago_areas_data)

print(chicago_area_df.info())
display(
    preview_geoms(chicago_area_df, n=5, width=50) 
)


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   area        77 non-null     float64 
 1   area_num_1  77 non-null     object  
 2   area_numbe  77 non-null     object  
 3   comarea     77 non-null     float64 
 4   comarea_id  77 non-null     float64 
 5   community   77 non-null     object  
 6   perimeter   77 non-null     float64 
 7   shape_area  77 non-null     float64 
 8   shape_len   77 non-null     float64 
 9   geometry    77 non-null     geometry
dtypes: float64(6), geometry(1), object(3)
memory usage: 6.1+ KB
None


/usr/local/lib/python3.7/site-packages/geopandas/geodataframe.py:1350: UserWarning: Geometry column does not contain geometry.
  warnings.warn("Geometry column does not contain geometry.")


,area,area_num_1,area_numbe,comarea,comarea_id,community,perimeter,shape_area,shape_len,geometry
0,0.0,35,35,0.0,0.0,DOUGLAS,0.0,4.600462e+07,31027.054510,"POLYGON ((-87.609141 41.844693, -87.609149 41.8446..."
1,0.0,36,36,0.0,0.0,OAKLAND,0.0,1.691396e+07,19565.506153,"POLYGON ((-87.592153 41.816929, -87.592308 41.8169..."
2,0.0,37,37,0.0,0.0,FULLER PARK,0.0,1.991670e+07,25339.089750,"POLYGON ((-87.628798 41.801893, -87.628794 41.8017..."
3,0.0,38,38,0.0,0.0,GRAND BOULEVARD,0.0,4.849250e+07,28196.837157,"POLYGON ((-87.606708 41.816814, -87.606705 41.8165..."
4,0.0,39,39,0.0,0.0,KENWOOD,0.0,2.907174e+07,23325.167906,"POLYGON ((-87.592153 41.816929, -87.592149 41.8168..."


(Utilice este espacio para describir su procedimiento).

### 2.   Análisis distribución del crimen por barrios

#### 2.1.  Genere una tabla descriptiva donde se muestra el número total de delitos, el número total de robos y el número total de homicidios, y como porcentaje de total por barrios. La tabla debe contener ademas una fila final donde se muestre el total para la ciudad. Describa los resultados que obtiene.


In [ ]:
# Utilice este espacio para escribir el código.

(Utilice este espacio para describir el procedimiento, análisis, y conclusiones)

#### 2.2. Genere una gráfica de dispersión entre el total de homicidios y robos por barrios. Incluya en la gráfica la recta de regresión que mejor ajusta a esos datos. Describa los resultados que obtiene.

In [ ]:
# Utilice este espacio para escribir el código.

(Utilice este espacio para describir el procedimiento, análisis, y conclusiones)

### 3. Distribución espacial del delito

#### 3.1 Distribución respecto al centro de la ciudad

Tomando como centro de la ciudad las coordenadas (-87.627800, 41.881998), estime funciones de densidad que muestren gráficamente el gradiente del total de robos, y homicidios, como función de la distancia al centro de la ciudad. Explique cómo midió las distancias incluyendo que medida de distancia utilizó. Para elegir el ancho de banda y la función de kernel más apropiados utilice validación cruzada usando todas las opciones posibles de kernel. Describa los resultados que obtiene.

In [ ]:
# Utilice este espacio para escribir el código.

(Utilice este espacio para describir el procedimiento, análisis, y conclusiones)

### 3.2 Puntos calientes en la ciudad

Usando `statsmodels` implemente la estimación de densidad bivariada para el total de robos y el total de homicidios. Muestre los resultados usando curvas de nivel en una visualización interactiva. Compare los resultados de estimar usando los anchos de banda: `normal_reference` y `cv_ml`. Explique en que consisten ambas formas de estimar el ancho de banda. Comente sobre los puntos calientes encontrados bajo ambos métodos y su ubicación en la ciudad. (Esto puede tomar mucho tiempo y requerir mucha capacidad computacional, puede aprovechar los recursos de [Google Colab](https://colab.research.google.com/))

In [ ]:
# Utilice este espacio para escribir el código.

(Utilice este espacio para describir el procedimiento, análisis, y conclusiones)

## 4. Explicando la ubicación del delito

El objetivo de este punto es encontrar posibles correlaciones  entre el crimen y características de la ciudad. Para ello, utilice los datos de OpenStreetMap y explore si existe una correlación entre el porcentaje del área de la comunidad  dedicado a tiendas (`retail`)  y comercios (`commercial`) y el número total de robos y homicidios en esa comunidad. Ofrezca una explicación intuitiva de por qué cree que aparecen estas correlaciones. (Esto puede tomar mucho tiempo y requerir mucha capacidad computacional, puede aprovechar los recursos de [Google Colab](https://colab.research.google.com/))

In [ ]:
# Utilice este espacio para escribir el código.

(Utilice este espacio para describir el procedimiento, análisis, y conclusiones)